In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# 建立向量索引
# ToDo: 配置向量模型
def build_index(pdf_path: str) -> VectorStoreIndex:
    print(f"📄 Loading and indexing {pdf_path}...")
    reader = SimpleDirectoryReader(input_files=[pdf_path])
    docs = reader.load_data()
    index = VectorStoreIndex.from_documents(docs)
    print(f"✅ Indexed {len(docs)} document chunk(s)")
    return index

In [ ]:
import os

from llama_index.llms.openai import OpenAI
from llama_index.core.memory import ChatMemoryBuffer

model = os.getenv('MODEL_NAME')

# 可选的 chat_mode 行为：
# "context"     → 先检索上下文，再让 LLM 基于上下文回答（最常用）
# "auto"        → 先判断是否需要检索（用 LLM 判断是开放问答还是基于文档）
# "refine"      → 对多个检索到的文档逐个细化回答（更准确但更慢）
# "tree_push"   → 为 RAG 构建知识图谱结构
# "summary"     → 对文档做总结
# "condense"    → 把多轮对话压缩成单轮检索查询（解决长对话上下文膨胀）

def create_chat_engine(index: VectorStoreIndex):
    llm = OpenAI(model=model, temperature=0)
    memory = ChatMemoryBuffer.from_defaults(token_limit=4096)
    chat_engine = index.as_chat_engine(
        chat_mode="context",
        llm=llm,
        memory=memory,
        verbose=False,
    )

    return chat_engine
        

In [ ]:
def create_query_engine(index: VectorStoreIndex):
    query_engine = index.as_query_engine(similarity_top_k=5)
    return query_engine

In [ ]:
question = "3季度GDP增长情况怎么样？"
index = build_index(pdf_path="/Users/linqibin/Documents/code/agents/my-agents/apps/backend/documents/pdf/中国经济金融展望报告.pdf")

In [ ]:
chat_engine = create_chat_engine(index=index)
query_engine = create_query_engine(index=index)

In [ ]:

response = chat_engine.chat(question)
print(f"\nAgent: {response.response}\n")

In [ ]:
result = query_engine.query(question)
print(result.response)